In [22]:
"""
Process extracted metadata following the format from submission samples
"""
import pandas as pd

df_extracted = pd.read_csv("test_predictions.csv")

In [ ]:
import re

def clean_lot_number(lot_num):
    """Clean and format lot numbers to XX.XX.XX.XXX format"""
    # Replace / and - with .
    cleaned = str(lot_num).replace('/', '.').replace('-', '.')

    # Remove any non-alphanumeric characters except dots
    cleaned = re.sub(r'[^a-zA-Z0-9.]', '', cleaned)

    # Split by dots and take first 4 parts, each capped at max length
    parts = cleaned.split('.')
    formatted_parts = []

    for i, part in enumerate(parts[:4]):  # Take max 4 parts
        if i == 0:  # First part: max 2 digits
            formatted_parts.append(part[:2])
        elif i == 1:  # Second part: max 2 digits
            formatted_parts.append(part[:2])
        elif i == 2:  # Third part: max 2 digits
            formatted_parts.append(part[:2])
        elif i == 3:  # Fourth part: max 3 digits
            formatted_parts.append(part[:3])

    return '.'.join(formatted_parts)

# Store original lot numbers before cleaning
original_lot_nums = df_extracted['LT Num'].copy()

# Apply cleaning
df_extracted['LT Num'] = df_extracted['LT Num'].apply(clean_lot_number)

# Find and display changes
changes = []
for idx, (original, cleaned) in enumerate(zip(original_lot_nums, df_extracted['LT Num'])):
    if str(original) != str(cleaned):
        changes.append({
            'Index': idx,
            'ID': df_extracted.iloc[idx]['ID'] if 'ID' in df_extracted.columns else idx,
            'Original': original,
            'Cleaned': cleaned
        })

print("Lot numbers cleaned")
print(f"Sample: {df_extracted['LT Num'].head(10).tolist()}")
print(f"\nTotal lot numbers: {len(df_extracted)}")
print(f"Changed: {len(changes)}")
print(f"Unchanged: {len(df_extracted) - len(changes)}")

if changes:
    print(f"\nFirst 20 changes:")
    changes_df = pd.DataFrame(changes[:20])
    print(changes_df.to_string(index=False))

    if len(changes) > 20:
        print(f"\n... and {len(changes) - 20} more changes")
else:
    print("\nNo changes needed - all lot numbers were already in correct format")

In [ ]:
def clean_unit_of_measurement(unit):
    """
    Automatically clean and standardize unit of measurement.
    Detects 'h', 'a' patterns for hectares -> 'ha'
    Everything else defaults to -> 'sq m'
    Works regardless of spacing, dots, or special characters.
    """
    if pd.isna(unit):
        return unit

    # Convert to string and lowercase, remove extra whitespace
    unit_str = str(unit).strip().lower()

    # Remove all dots, commas, and special characters but keep letters and spaces
    cleaned = re.sub(r'[.,\-_/\\]', ' ', unit_str)

    # Remove multiple spaces
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()

    # Extract only letters (ignore numbers and other characters)
    letters_only = ''.join(c for c in cleaned if c.isalpha() or c.isspace())

    # Check for hectare patterns: h, a (in any spacing)
    # Pattern: h followed by optional 'and'/'&' followed by a
    ha_pattern = r'h\s*(?:and)?\s*[&]?\s*a(?:\s|$)'
    if re.search(ha_pattern, letters_only):
        return 'ha'

    # Alternative: just 'ha' without spaces
    if 'ha' in letters_only.replace(' ', ''):
        return 'ha'

    # Default: everything else is square meters
    return 'sq m'

# Apply cleaning to Unit of Measurement column
df_extracted['Unit of Measurement'] = df_extracted['Unit of Measurement'].apply(clean_unit_of_measurement)

print("Unit of Measurement cleaned")
print(f"Unique units after cleaning: {df_extracted['Unit of Measurement'].unique()}")
print(f"\nSample results:")
df_extracted[['ID', 'Unit of Measurement']].head(5)

In [25]:
from dateutil import parser

# Load CSV

def normalize_date(s):
    if pd.isna(s):
        return s
    # remove ordinal suffixes: 1st, 2nd, 3rd, 4th, ...
    s = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', s)
    # parse and format
    return parser.parse(s, dayfirst=True).strftime("%Y-%m-%d")

df_extracted["Certified date"] = df_extracted["Certified date"].apply(normalize_date)

In [26]:
def extract_parish_from_address(address):
    """
    Extract parish (St./Saint Name pattern) from address.
    Returns the parish name formatted as 'St. Name' with proper capitalization.
    Handles both "St." and "Saint" patterns, with or without comma, with or without trailing period.
    Also handles patterns without space like "ST.PHILIP" or "STPHILIP".
    """
    if pd.isna(address):
        return ""

    address_str = str(address).strip().lower()

    # Pattern 1: Match "saint [name]" at the end (with or without comma, with or without trailing period)
    saint_pattern = r',?\s*saint\.?\s+([A-Za-z]+)\.?\s*$'
    match = re.search(saint_pattern, address_str)

    if match:
        parish_name = match.group(1).capitalize()
        return f"St. {parish_name}"

    # Pattern 2: Match "st. [name]" or "st [name]" at the end (with or without comma, with or without trailing period)
    st_pattern = r',?\s*st\.?\s+([A-Za-z]+)\.?\s*$'
    match = re.search(st_pattern, address_str)

    if match:
        parish_name = match.group(1).capitalize()
        return f"St. {parish_name}"

    # Pattern 3: Match "st.[name]" or "st[name]" without space (like ST.PHILIP or STPHILIP)
    st_nospace_pattern = r',?\s*st\.?([A-Za-z]+)\.?\s*$'
    match = re.search(st_nospace_pattern, address_str)

    if match:
        parish_name = match.group(1).capitalize()
        return f"St. {parish_name}"

    return ""

In [ ]:
def remove_parish_from_address(address):
    """
    Remove parish (St./Saint Name pattern) from the end of address.
    Handles both "St." and "Saint" patterns, with or without comma, and with or without trailing period.
    Also handles patterns without space like "ST.PHILIP" or "STPHILIP".
    """
    if pd.isna(address):
        return address

    address_str = str(address).strip()

    # Pattern to match and remove "Saint [Name]", "St. [Name]", "St.[Name]", or "St[Name]" at the end (case insensitive, with or without trailing period)
    parish_pattern = r',?\s*(?:saint|st)\.?\s*[A-Za-z]+\.?\s*$'
    cleaned_address = re.sub(parish_pattern, '', address_str, flags=re.IGNORECASE)

    return cleaned_address.strip()

# Update Address column to remove parish
df_extracted['Address'] = df_extracted['Address'].apply(remove_parish_from_address)

print("Parish extracted and Address cleaned")
print(f"\nUnique parishes: {df_extracted['Parish'].unique()}")
print(f"\nSample results:")
df_extracted[['ID', 'Address', 'Parish']].head(10)

In [ ]:
import re
import numpy as np

def clean_land_surveyor_names(names_array):
    """
    Clean land surveyor names by removing middle initials while preserving:
    - Names that start with initials (like H.A. King)
    - Names with nicknames in quotes
    - Names with compound elements like St. Clair
    - Professional designations like JP
    """

    def clean_single_name(name):
        if pd.isna(name) or not name or name.strip() == '':
            return name

        name = str(name).strip()

        # Skip names that start with initials (like "H.A King" or "H.A. King")
        if re.match(r'^[A-Z]\.?\s*[A-Z]\.?\s+', name):
          return name

        # Skip names with quotes (nicknames like D.C "Vallan" Franklin JP)
        if '"' in name:
            return name

        # Skip single names (like "Simba")
        if len(name.split()) <= 1:
            return name

        # Protect "St." in compound names like "Michelle E. St. Clair"
        name_protected = name.replace(' St. ', ' PROTECTED_ST ')

        # Remove middle initials patterns:
        # 1. Single initials: "Lennox J Reid" → "Lennox Reid"
        name_cleaned = re.sub(r'\s+[A-Z]\.?\s+', ' ', name_protected)

        # 2. Multiple initials: "Jamal K.L. Gaskin" → "Jamal Gaskin"
        name_cleaned = re.sub(r'\s+[A-Z]\.[A-Z]\.?\s+', ' ', name_cleaned)

        # 3. Space-separated initials: "Lee B S Brathwaite" → "Lee Brathwaite"
        name_cleaned = re.sub(r'\s+[A-Z]\s+[A-Z]\s+', ' ', name_cleaned)

        # 4. Complex patterns like "Lee B.S Brathwaite" or "Sekani H.C Franklin"
        name_cleaned = re.sub(r'\s+[A-Z]\.[A-Z]\s+', ' ', name_cleaned)

        # 5. Handle remaining single initials that might be left
        name_cleaned = re.sub(r'\s+[A-Z]\.?\s+', ' ', name_cleaned)

        # Restore protected "St."
        name_cleaned = name_cleaned.replace(' PROTECTED_ST ', ' St. ')

        # Clean up multiple spaces and trim
        name_cleaned = re.sub(r'\s+', ' ', name_cleaned).strip()

        return name_cleaned

    # Apply cleaning to each name in the array
    if isinstance(names_array, np.ndarray):
        return np.array([clean_single_name(name) for name in names_array])
    elif isinstance(names_array, (list, pd.Series)):
        return [clean_single_name(name) for name in names_array]
    else:
        return clean_single_name(names_array)

# Create working copy
data_expanded = df_extracted.copy()

# Apply to both name columns
data_expanded['Land Surveyor'] = clean_land_surveyor_names(data_expanded['Land Surveyor'])
data_expanded['Surveyed For'] = clean_land_surveyor_names(data_expanded['Surveyed For'])

print("Names cleaned (middle initials removed)")

In [29]:
def clean_target_survey(text: str) -> str:
    """Lowercase, remove periods and commas, normalize spaces"""
    text = text.lower()
    text = re.sub(r"[.,]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [ ]:
def format_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Create TargetSurvey and keep only required columns"""
    df["TargetSurvey"] = (
        df["Land Surveyor"].astype(str).str.strip() + " " +
        df["Surveyed For"].astype(str).str.strip() + " " +
        df["Address"].astype(str).str.strip()
    ).apply(clean_target_survey)

    columns_to_keep = [
        'ID', 'TargetSurvey', 'Certified date', 'Total Area',
        'Unit of Measurement', 'Parish', 'LT Num'
    ]
    return df[columns_to_keep]

final_data = format_dataset(data_expanded)

print("Final dataset created")
print(f"\nColumns: {final_data.columns.tolist()}")
print(f"\nTotal records: {len(final_data)}")
final_data.head(10)

In [31]:
# Save to CSV
final_data.to_csv('text_extraction_final.csv', index=False)

In [32]:
final_data

,ID,TargetSurvey,Certified date,Total Area,Unit of Measurement,Parish,LT Num
0,7703-078,ronald greene nero millington lot 636 ruby dev...,2020-09-14,430.8,sq m,Barbados,77.03.07.015
1,8606-095,lennox reid collymore taylor lot 7a mangrove p...,2021-10-14,428.7,sq m,Barbados,86.06.07.090
2,7703-064,ronald greene frank frank lot 632 ruby develop...,2019-01-18,661.8,sq m,St. Michael,77.03.08.034
3,7703-101,michael banfield smith roachford lot 114 ruby ...,2022-08-27,527.3,sq m,St. Philip,77.03.02.054
4,7707-114,michael hutchinson liam worrell lot 224 union ...,2013-02-07,540.4,sq m,St. Philip,77.07.06.028
...,...,...,...,...,...,...,...
214,BYLS-083,simba nala lot 8 gemswick,2023-12-15,1780.5,sq m,St. Philip,94.03.02.020
215,BYLS-084,simba don carlo lot 88 castle heights development,2024-09-30,518.2,sq m,St. Philip,67.16.05.016
216,BYLS-114,lennox reid omar webb lot 3 st martins,2025-01-17,1265.9,sq m,St. Philip,86.11.03.026
217,BYLS-115,"lennox reid donna kerr lot 54 ""st martins terr...",2025-03-12,558.1,sq m,St. Philip,86.07.08.054
